In [1]:
!pip install facenet-pytorch streamlit

In [2]:
import streamlit as st
import torch
import torch.nn as nn
import cv2
import os
import json
import hashlib
import tempfile
import numpy as np
from PIL import Image
from torchvision import models, transforms, datasets
from facenet_pytorch import MTCNN

st.set_page_config(
    page_title="AegisFace - Deepfake Forensics Studio",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown("""
    <style>
    .main { background-color: #0E1117; }
    .stMetric { background-color: #1E222D; padding: 15px; border-radius: 10px; border: 1px solid #2E3440; }
    .status-real { background-color: #1b4332; color: #2ecc71; padding: 14px; border-radius: 8px; font-weight: bold; text-align: center; font-size: 18px; }
    .status-fake { background-color: #4a0e17; color: #e74c3c; padding: 14px; border-radius: 8px; font-weight: bold; text-align: center; font-size: 18px; }
    .stButton>button { width: 100%; border-radius: 8px; height: 42px; background-color: #4F46E5; color: white; font-weight: 600; }
    </style>
""", unsafe_allow_html=True)

USER_DB = "users.json"

def hash_password(password):
    return hashlib.sha256(password.encode()).hexdigest()

def load_users():
    if not os.path.exists(USER_DB):
        with open(USER_DB, "w") as f:
            json.dump({"admin": hash_password("admin123")}, f)
    with open(USER_DB, "r") as f:
        return json.load(f)

def save_user(username, password):
    users = load_users()
    users[username] = hash_password(password)
    with open(USER_DB, "w") as f:
        json.dump(users, f)

if "authenticated" not in st.session_state:
    st.session_state.authenticated = False
if "username" not in st.session_state:
    st.session_state.username = ""

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

@st.cache_resource
def init_detector():
    mtcnn = MTCNN(keep_all=False, post_process=False, select_largest=True, device=device)
    model = models.vit_b_16()
    model.heads.head = nn.Linear(model.heads.head.in_features, 2)

    real_idx = 1
    weights_path = "weights/model_complete.pth"
    if not os.path.exists(weights_path):
        weights_path = "weights/vit_deepfake.pth"

    if os.path.exists(weights_path):
        checkpoint = torch.load(weights_path, map_location=device)
        if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
            model.load_state_dict(checkpoint["state_dict"])
            class_mapping = checkpoint.get("class_to_idx", {"fake": 0, "real": 1})
            real_idx = class_mapping.get("real", class_mapping.get("Real", 1))
        else:
            model.load_state_dict(checkpoint)

    model.to(device).eval()
    return mtcnn, model, real_idx

mtcnn, model, REAL_IDX = init_detector()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

if not st.session_state.authenticated:
    st.title("🛡️ AegisFace Platform")
    st.caption("AI-Powered Deepfake Detection Suite")

    tab1, tab2 = st.tabs(["🔐 Sign In", "📝 Register"])
    with tab1:
        u = st.text_input("Username", key="login_user")
        p = st.text_input("Password", type="password", key="login_pwd")
        if st.button("Log In"):
            users = load_users()
            if u in users and users[u] == hash_password(p):
                st.session_state.authenticated = True
                st.session_state.username = u
                st.rerun()
            else:
                st.error("Invalid credentials.")
    with tab2:
        nu = st.text_input("New Username", key="reg_user")
        npw = st.text_input("New Password", type="password", key="reg_pwd")
        cpw = st.text_input("Confirm Password", type="password", key="reg_pwd_conf")
        if st.button("Register"):
            users = load_users()
            if not nu or not npw or npw != cpw or nu in users:
                st.error("Registration failed. Check password match or existing user.")
            else:
                save_user(nu, npw)
                st.success("Account created! Log in above.")
    st.stop()

with st.sidebar:
    st.title("🛡️ AegisFace AI")
    st.caption(f"Operator: {st.session_state.username}")
    st.markdown("---")
    st.text(f"Device: {device.type.upper()}")
    st.text("Model: ViT-B/16 Engine")
    st.text(f"Target Real Index: Class {REAL_IDX}")
    st.markdown("---")
    if st.button("Logout"):
        st.session_state.authenticated = False
        st.session_state.username = ""
        st.rerun()

st.title("Video & Image Deepfake Forensics")
st.markdown("Upload a video (`.mp4`) or image (`.jpg`, `.png`) to perform facial spatial verification.")

uploaded_file = st.file_uploader("Drop Media File", type=["jpg", "png", "jpeg", "mp4"])

def extract_and_predict(pil_images):
    tensors = []
    crops = []

    for img in pil_images:
        boxes, _ = mtcnn.detect(img)
        if boxes is not None and len(boxes) > 0:
            box = boxes[0].astype(int)
            w, h = img.size
            x1, y1 = max(0, box[0]), max(0, box[1])
            x2, y2 = min(w, box[2]), min(h, box[3])

            if x2 > x1 and y2 > y1:
                crop = img.crop((x1, y1, x2, y2))
                tensors.append(transform(crop))
                crops.append(crop)

    if not tensors:
        return None, []

    batch = torch.stack(tensors).to(device)
    with torch.no_grad():
        if device.type == 'cuda':
            with torch.cuda.amp.autocast():
                logits = model(batch)
        else:
            logits = model(batch)

        probs = torch.nn.functional.softmax(logits, dim=1)[:, REAL_IDX].cpu().numpy()

    return probs, crops

if uploaded_file:
    col1, col2 = st.columns([1, 1])

    if uploaded_file.name.lower().endswith(('.jpg', '.jpeg', '.png')):
        img = Image.open(uploaded_file).convert("RGB")
        with col1:
            st.image(img, caption="Target Input Image", use_container_width=True)
        with col2:
            st.subheader("Image Analysis Platform")
            if st.button("🔍 Submit Image for Scan"):
                with st.spinner("Processing face crop..."):
                    probs, crops = extract_and_predict([img])

                if probs is not None and len(probs) > 0:
                    real_prob = probs[0]
                    is_real = real_prob > 0.5
                    conf = real_prob if is_real else (1 - real_prob)

                    st.image(crops[0], caption="Extracted Face Box", width=180)

                    m1, m2 = st.columns(2)
                    m1.metric("Assessment", "REAL" if is_real else "DEEPFAKE")
                    m2.metric("Confidence", f"{conf * 100:.1f}%")

                    if is_real:
                        st.markdown('<div class="status-real">✅ VERDICT: REAL (Authentic)</div>', unsafe_allow_html=True)
                    else:
                        st.markdown('<div class="status-fake">🚨 VERDICT: DEEPFAKE DETECTED</div>', unsafe_allow_html=True)
                else:
                    st.warning("No faces detected in the provided image.")

    elif uploaded_file.name.lower().endswith('.mp4'):
        with tempfile.NamedTemporaryFile(delete=False, suffix=".mp4") as tfile:
            tfile.write(uploaded_file.read())
            vid_path = tfile.name

        with col1:
            st.video(vid_path)

        with col2:
            st.subheader("Video Analysis Platform")
            if st.button("🚀 Submit Video for Scan"):
                with st.spinner("Decoding video frames with OpenCV..."):
                    cap = cv2.VideoCapture(vid_path)
                    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                    sample_count = 24
                    step = max(1, total_frames // sample_count)

                    pil_frames = []
                    frame_idx = 0

                    while cap.isOpened() and len(pil_frames) < sample_count:
                        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
                        ret, frame = cap.read()
                        if not ret or frame is None:
                            break

                        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                        pil_frames.append(Image.fromarray(rgb_frame))
                        frame_idx += step

                    cap.release()

                    probs, crops = extract_and_predict(pil_frames)

                if probs is not None and len(probs) > 0:
                    avg_real_prob = np.mean(probs)
                    is_real = avg_real_prob > 0.5
                    conf = avg_real_prob if is_real else (1 - avg_real_prob)

                    st.markdown("**Sampled Facial Crops:**")
                    st.image(crops[:6], width=80)

                    m1, m2 = st.columns(2)
                    m1.metric("Video Assessment", "REAL" if is_real else "DEEPFAKE")
                    m2.metric("Average Confidence", f"{conf * 100:.1f}%")

                    if is_real:
                        st.markdown(f'<div class="status-real">✅ VERDICT: REAL VIDEO ({len(probs)} Frames Validated)</div>', unsafe_allow_html=True)
                    else:
                        st.markdown(f'<div class="status-fake">🚨 VERDICT: DEEPFAKE DETECTED ({len(probs)} Frames Validated)</div>', unsafe_allow_html=True)
                else:
                    st.warning("Could not isolate face bounding boxes from sampled frames.")

                if os.path.exists(vid_path):
                    os.remove(vid_path)

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
2026-07-20 21:17:53.662 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-20 21:17:53.664 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-20 21:17:53.783 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-07-20 21:17:53.785 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-20 21:17:53.786 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-20 21: